In [ ]:
%pip install requests dotenv

In [ ]:
import asyncio
import os
from dotenv import load_dotenv
import aiohttp

# Корректно и жестко определяем корень проекта для Windows среды Jupyter
try:
    notebook_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    notebook_dir = os.getcwd()

if os.path.basename(notebook_dir) == 'notebooks':
    project_root = os.path.dirname(notebook_dir)
else:
    project_root = notebook_dir

# Загружаем переменные окружения
dotenv_path = os.path.join(project_root, 'scripts', '.env')
load_dotenv(dotenv_path=dotenv_path)
WB_TOKEN = os.getenv("WB_TOKEN_ANALYTICS")

async def test_wb_connection():
    if not WB_TOKEN:
        print("Ошибка: Переменная WB_TOKEN не найдена в .env!")
        print(f"Попытка прочитать файл по пути: {os.path.abspath(dotenv_path)}")
        return

    # GET-метод возвращает список ранее созданных задач генерации отчетов
    url = "https://seller-analytics-api.wildberries.ru/api/v2/nm-report/downloads"
    
    headers = {
        "Authorization": WB_TOKEN,
        "accept": "application/json"
    }
    
    async with aiohttp.ClientSession() as session:
        try:
            # Используем GET для безопасной проверки связи и получения списка
            async with session.get(url, headers=headers) as response:
                print(f"Статус HTTP-ответа: {response.status}")
                
                if response.status == 200:
                    data = await response.json()
                    print("Успешно подключено к API Аналитики!")
                    # Выводим весь ответ, чтобы изучить ключи (data, error, meta и т.д.)
                    print("Полная структура ответа от WB:")
                    print(data)
                elif response.status == 401:
                    print("Ошибка 401: Неверный токен аналитики.")
                else:
                    text = await response.text()
                    print(f"Ответ сервера (Код {response.status}): {text}")
        except Exception as e:
            print(f"Произошла ошибка при обработке данных: {e}")

await test_wb_connection()


Статус HTTP-ответа: 200
Успешно подключено к API Аналитики!
Полная структура ответа от WB:
{'data': []}

In [ ]:
import base64
import json
from datetime import datetime, timezone

def inspect_wb_token(token: str, token_name: str = "WB_TOKEN") -> dict:
    """
    Выполняет pre-flight декодирование JWT токена без внешних библиотек.
    Проверяет TTL и возвращает метаданные для логирования.
    """
    if not token:
        print(f"[ERROR] Токен {token_name} пуст или отсутствует")
        return {"status": "missing"}
        
    try:
        # JWT состоит из трех частей, разделенных точкой: Header.Payload.Signature
        parts = token.split('.')
        if len(parts) != 3:
            print(f"[ERROR] Неверный формат токена {token_name}. Ожидалось 3 части JWT.")
            return {"status": "invalid_format"}
            
        payload_b64 = parts[1]
        
        # Добавляем необходимое дополнение (padding) для корректного b64-декодирования
        rem = len(payload_b64) % 4
        if rem > 0:
            payload_b64 += '=' * (4 - rem)
            
        # Декодируем байты и парсим JSON
        payload_bytes = base64.urlsafe_b64decode(payload_b64)
        payload_data = json.loads(payload_bytes.decode('utf-8'))
        
        # Извлекаем timestamp истечения (поле 'exp')
        exp_timestamp = payload_data.get("exp")
        if not exp_timestamp:
            print(f"[WARNING] В токене {token_name} отсутствует поле 'exp'")
            return {"status": "unknown_ttl", "payload": payload_data}
            
        # Рассчитываем оставшееся время
        exp_datetime = datetime.fromtimestamp(exp_timestamp, tz=timezone.utc)
        current_datetime = datetime.now(timezone.utc)
        time_left = exp_datetime - current_datetime
        days_left = time_left.days
        
        # Логика предупреждений
        if days_left <= 0:
            print(f"[CRITICAL] Срок действия токена {token_name} ИСТЕК!")
            status = "expired"
        elif days_left <= 5: # 120-часовой карантин согласно ТЗ
            print(f"[ALERT] Токен {token_name} находится в зоне 120-часового карантина! Около {days_left} дней до отключения.")
            status = "quarantine"
        else:
            print(f"[INFO] Токен {token_name} валиден. Оставшийся срок действия: {days_left} дней.")
            status = "valid"
            
        return {
            "status": status,
            "days_left": days_left,
            "expires_at": exp_datetime.isoformat(),
            "payload_keys": list(payload_data.keys()) # Безопасно логируем структуру без самих данных
        }
        
    except Exception as e:
        print(f"[ERROR] Не удалось распарсить токен {token_name}: {str(e)}")
        return {"status": "error", "error": str(e)}

# --- БЛОК ТЕСТИРОВАНИЯ ---
# Подтягиваем один из ваших июньских токенов из окружения (убедитесь, что .env подгружен)
import os
from dotenv import load_dotenv

# Нам нужно явно передать путь, так как .env лежит в папке scripts
dotenv_path = os.path.join("scripts", ".env")
load_dotenv(dotenv_path=dotenv_path)

test_token = os.getenv("WB_TOKEN_STATISTICS")
token_info = inspect_wb_token(test_token, "WB_TOKEN_STATISTICS")


[INFO] Токен WB_TOKEN_STATISTICS валиден. Оставшийся срок действия: 75 дней.

In [ ]:
import os
from dotenv import load_dotenv

# Метод 1: Явно задаем абсолютный путь от корня проекта, который мы знаем из контекста
root_dir = r"D:\Documents\Dev\projects\new-bi-project"
dotenv_path = os.path.join(root_dir, "scripts", ".env")

print(f"[DEBUG] Проверяем существование файла по пути: {dotenv_path}")
print(f"[DEBUG] Файл существует? {os.path.exists(dotenv_path)}")

# Загружаем переменные
load_dotenv(dotenv_path=dotenv_path)

# Проверяем, что прочиталось
test_token = os.getenv("WB_TOKEN_STATISTICS")
print(f"[DEBUG] Токен найден в .env? {test_token is not None}")

# Если токен найден, запускаем нашу функцию проверки
if test_token:
    token_info = inspect_wb_token(test_token, "WB_TOKEN_STATISTICS")


[DEBUG] Проверяем существование файла по пути: D:\Documents\Dev\projects\new-bi-project\scripts\.env
[DEBUG] Файл существует? True
[DEBUG] Токен найден в .env? True
[INFO] Токен WB_TOKEN_STATISTICS валиден. Оставшийся срок действия: 75 дней.

In [ ]:
import os
import time
import random
import uuid
import json
import logging
from datetime import datetime, timezone
import requests

def mask_token(token: str) -> str:
    """Маскирует токен, оставляя видимыми только первые и последние 8 символов."""
    if not token or len(token) < 20:
        return "***hidden***"
    return f"{token[:8]}...{token[-8:]}"

def safe_request(method: str, url: str, headers: dict = None, params: dict = None, json_data: dict = None, timeout: int = 60, max_retries: int = 4) -> requests.Response:
    """
    Отказоустойчивый HTTP-транспорт для Wildberries API.
    Реализует адаптивный ретрай 429 по X-Ratelimit-Retry, Jitter ±25%, обработку 409 кодов,
    сквозные request_id и структурированное JSON-логирование с маскированием секретов.
    """
    headers = headers.copy() if headers else {}
    
    # Генерируем сквозной идентификатор запроса на стороне клиента
    client_req_id = str(uuid.uuid4())
    headers["client_request_id"] = client_req_id # для отслеживания в нашей системе
    
    # Маскируем авторизационный токен для безопасного логирования
    auth_header = headers.get("Authorization", "")
    masked_auth = mask_token(auth_header)
    
    base_wait_409 = 5.0 # Базовая пауза для 409 ошибок
    
    for attempt in range(1, max_retries + 1):
        log_payload = {
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "level": "INFO",
            "client_request_id": client_req_id,
            "attempt": attempt,
            "method": method.upper(),
            "url": url,
            "params": params,
            "token": masked_auth
        }
        
        try:
            # Выполняем сетевой запрос
            resp = requests.request(
                method=method.upper(),
                url=url,
                headers=headers,
                params=params,
                json=json_data,
                timeout=timeout
            )
            
            # Извлекаем request_id, который нам присвоил сервер WB
            wb_req_id = resp.headers.get("x-request-id", "unknown_wb_id")
            log_payload["wb_request_id"] = wb_req_id
            log_payload["status_code"] = resp.status_code
            
            # Успешный запрос или штатная ошибка (кроме лимитов и блокировок)
            if resp.status_code not in (429, 409):
                print(json.dumps(log_payload)) # Имитируем структурированную JSON-строку в логе
                return resp
                
            # Обработка 429 Too Many Requests (Превышение лимитов)
            if resp.status_code == 429:
                log_payload["level"] = "WARNING"
                retry_after = resp.headers.get('X-Ratelimit-Retry')
                
                if retry_after:
                    # Адаптивное ожидание по заголовку WB
                    wait_time = float(retry_after)
                else:
                    # Дефолтное жесткое ожидание, если заголовок не пришел
                    wait_time = 60.0
                    
                # Добавляем Jitter ±25% к вычисленному времени
                jitter = wait_time * random.uniform(-0.25, 0.25)
                final_wait = max(0.5, wait_time + jitter)
                
                log_payload["message"] = f"API Rate Limit 429. Ожидание {final_wait:.2f} сек (адаптивно: {retry_after}, jitter применен)."
                print(json.dumps(log_payload))
                
                if attempt < max_retries:
                    time.sleep(final_wait)
                    continue
                else:
                    break
                    
            # Обработка 409 Conflict (Параллельные запросы или блокировки)
            if resp.status_code == 409:
                log_payload["level"] = "WARNING"
                # Экспоненциальный рост паузы для 409 кодов + jitter
                wait_time = base_wait_409 * (2 ** (attempt - 1))
                jitter = wait_time * random.uniform(-0.25, 0.25)
                final_wait = max(1.0, wait_time + jitter)
                
                log_payload["message"] = f"Conflict 409. Пауза перед повтором: {final_wait:.2f} сек."
                print(json.dumps(log_payload))
                
                if attempt < max_retries:
                    time.sleep(final_wait)
                    continue
                else:
                    break
                    
        except requests.exceptions.RequestException as e:
            log_payload["level"] = "ERROR"
            log_payload["error"] = str(e)
            # При сетевой ошибке делаем базовый ретрай с нарастающей паузой
            wait_time = 5.0 * attempt
            log_payload["message"] = f"Сетевая ошибка. Повтор через {wait_time} сек."
            print(json.dumps(log_payload))
            
            if attempt < max_retries:
                time.sleep(wait_time)
                continue
            else:
                raise Exception(f"Не удалось выполнить запрос после {max_retries} попыток из-за сетевых ошибок.")
                
    raise Exception(f"Превышено число попыток ({max_retries}). Статус-код ответа: {log_payload.get('status_code')}")

# --- БЛОК ТЕСТИРОВАНИЯ ТРАНСПОРТА ---
# Выполним тестовый запрос к публичному эндпоинту WB (для теста токен не важен, нам нужно проверить формирование логов и отправку)
try:
    headers = {"Authorization": os.getenv("WB_TOKEN_STATISTICS", "")}
    # Используем любой валидный URL, например, эндпоинт заказов, чтобы посмотреть на структуру ответа или ошибку авторизации (главное проверить транспорт)
    url = "https://statistics-api.wildberries.ru/api/v1/supplier/orders"
    params = {"dateFrom": "2026-10-24T00:00:00", "flag": 1}
    
    print("--- Запуск теста safe_request ---")
    response = safe_request("GET", url, headers=headers, params=params, timeout=10)
    print(f"--- Тест завершен. Код ответа сервера: {response.status_code} ---")
except Exception as ex:
    print(f"--- Тест завершен исключением: {ex} ---")


Результат теста: Отказоустойчивый транспорт safe_request успешно протестирован. Выявлены и обработаны 2 ошибки таймаута (Read timed out). На 3-й попытке получен ответ 200 OK с фиксацией wb_request_id. Маскирование токенов работает штатно.

In [ ]:
import os
import time
import random
import uuid
import json
import logging
import base64
from datetime import datetime, timezone, date, timedelta
import requests
from dotenv import load_dotenv

# Настраиваем логирование в соответствии с промышленным стандартом
logging.basicConfig(level=logging.INFO, format='%(message)s')
log = logging.getLogger(__name__)

# Инициализация путей окружения
ROOT_DIR = r"D:\Documents\Dev\projects\new-bi-project"
dotenv_path = os.path.join(ROOT_DIR, "scripts", ".env")
load_dotenv(dotenv_path=dotenv_path)

# Валидация наличия токенов в конфигурации
WB_TOKEN_STAT = os.getenv("WB_TOKEN_STATISTICS")
WB_TOKEN_CONTENT = os.getenv("WB_TOKEN_CONTENT")
WB_TOKEN_ANALYTICS = os.getenv("WB_TOKEN_ANALYTICS")

def mask_token(token: str) -> str:
    """Маскирует токен для безопасности логов."""
    if not token or len(token) < 20:
        return "***hidden***"
    return f"{token[:8]}...{token[-8:]}"

def write_json_log(level: str, client_req_id: str, method: str, url: str, status_code: int = None, wb_req_id: str = None, message: str = None, error: str = None, masked_token_str: str = None) -> None:
    log_payload = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "level": level,
        "client_request_id": client_req_id,
        "method": method,
        "url": url,
        "token": masked_token_str
    }
    if status_code: log_payload["status_code"] = status_code
    if wb_req_id: log_payload["wb_request_id"] = wb_req_id
    if message: log_payload["message"] = message
    if error: log_payload["error"] = error
    
    # flush=True заставит Jupyter выводить строку на экран МОМЕНТАЛЬНО
    print(json.dumps(log_payload), flush=True)

def inspect_wb_token(token: str, token_name: str) -> bool:
    """Pre-flight проверка TTL токена."""
    if not token:
        write_json_log("ERROR", "pre-flight", "NONE", "NONE", message=f"Токен {token_name} отсутствует")
        return False
    try:
        parts = token.split('.')
        if len(parts) != 3:
            write_json_log("ERROR", "pre-flight", "NONE", "NONE", message=f"Неверный формат JWT {token_name}")
            return False
            
        payload_b64 = parts[1]
        rem = len(payload_b64) % 4
        if rem > 0:
            payload_b64 += '=' * (4 - rem)
            
        payload_bytes = base64.urlsafe_b64decode(payload_b64)
        payload_data = json.loads(payload_bytes.decode('utf-8'))
        
        exp_timestamp = payload_data.get("exp")
        if not exp_timestamp:
            write_json_log("WARNING", "pre-flight", "NONE", "NONE", message=f"В токене {token_name} нет поля exp")
            return True
            
        exp_datetime = datetime.fromtimestamp(exp_timestamp, tz=timezone.utc)
        days_left = (exp_datetime - datetime.now(timezone.utc)).days
        
        if days_left <= 0:
            write_json_log("CRITICAL", "pre-flight", "NONE", "NONE", message=f"Срок действия токена {token_name} ИСТЕК")
            return False
        elif days_left <= 5:
            write_json_log("ALERT", "pre-flight", "NONE", "NONE", message=f"Токен {token_name} в зоне 120-часового карантина. Осталось дней: {days_left}")
        else:
            write_json_log("INFO", "pre-flight", "NONE", "NONE", message=f"Токен {token_name} валиден. Оставшийся TTL: {days_left} дней.")
            
        return True
    except Exception as e:
        write_json_log("ERROR", "pre-flight", "NONE", "NONE", error=str(e), message=f"Ошибка валидации токена {token_name}")
        return False

def safe_request(method: str, url: str, headers: dict = None, params: dict = None, json_data: dict = None, timeout: int = 60, max_retries: int = 4) -> requests.Response:
    """Отказоустойчивый HTTP-транспорт с адаптивным ретраем и джиттером."""
    headers = headers.copy() if headers else {}
    client_req_id = str(uuid.uuid4())
    headers["client_request_id"] = client_req_id
    
    masked_auth = mask_token(headers.get("Authorization", ""))
    base_wait_409 = 5.0
    
    for attempt in range(1, max_retries + 1):
        try:
            resp = requests.request(method=method.upper(), url=url, headers=headers, params=params, json=json_data, timeout=timeout)
            wb_req_id = resp.headers.get("x-request-id", "unknown_wb_id")
            
            if resp.status_code not in (429, 409):
                write_json_log("INFO", client_req_id, method, url, status_code=resp.status_code, wb_req_id=wb_req_id, masked_token_str=masked_auth)
                return resp
                
            if resp.status_code == 429:
                retry_after = resp.headers.get('X-Ratelimit-Retry')
                wait_time = float(retry_after) if retry_after else 60.0
                jitter = wait_time * random.uniform(-0.25, 0.25)
                final_wait = max(0.5, wait_time + jitter)
                
                write_json_log("WARNING", client_req_id, method, url, status_code=429, wb_req_id=wb_req_id, masked_token_str=masked_auth,
                               message=f"API Rate Limit 429. Ретрай через {final_wait:.2f} сек.")
                if attempt < max_retries:
                    time.sleep(final_wait)
                    continue
                break
                
            if resp.status_code == 409:
                wait_time = base_wait_409 * (2 ** (attempt - 1))
                jitter = wait_time * random.uniform(-0.25, 0.25)
                final_wait = max(1.0, wait_time + jitter)
                
                write_json_log("WARNING", client_req_id, method, url, status_code=409, wb_req_id=wb_req_id, masked_token_str=masked_auth,
                               message=f"Conflict 409. Ретрай через {final_wait:.2f} сек.")
                if attempt < max_retries:
                    time.sleep(final_wait)
                    continue
                break
                
        except requests.exceptions.RequestException as e:
            wait_time = 5.0 * attempt
            write_json_log("ERROR", client_req_id, method, url, masked_token_str=masked_auth, error=str(e),
                           message=f"Сетевая ошибка. Повтор через {wait_time} сек.")
            if attempt < max_retries:
                time.sleep(wait_time)
                continue
            raise Exception(f"Запрос провален после {max_retries} попыток сетевых ошибок.")
            
    raise Exception(f"Превышено число попыток ретрая ({max_retries}). Статус: {resp.status_code}")

def fetch_marketplace_orders_raw() -> list:
    """Выгружает сырые заказы из API Статистики WB за скользящее окно (последние 5 дней) через safe_request."""
    if not inspect_wb_token(WB_TOKEN_STAT, "WB_TOKEN_STATISTICS"):
        raise ValueError("Префлай-валидация токена статистики провалена.")
        
    headers = {"Authorization": WB_TOKEN_STAT}
    all_orders = []
    url = "https://statistics-api.wildberries.ru/api/v1/supplier/orders"
    
    start_date = date.today() - timedelta(days=5)
    end_date = date.today()
    
    current_date = start_date
    while current_date <= end_date:
        date_from_str = f"{current_date.isoformat()}T00:00:00"
        params = {"dateFrom": date_from_str, "flag": 1}
        
        resp = safe_request("GET", url, headers=headers, params=params, timeout=60)
        if resp.status_code == 400:
            write_json_log("ERROR", "pipeline", "GET", url, status_code=400, message=f"Ошибка 400: {resp.text}")
            break
            
        orders = resp.json()
        if isinstance(orders, list) and orders:
            all_orders.extend(orders)
            
        current_date += timedelta(days=1)
        # Адаптивный клиент убрал необходимость в жестких time.sleep(15) после каждого дня,
        # однако между днями оставляем микро-паузу безопасности в 1 секунду
        time.sleep(1.0)
        
    return all_orders

# Метод fetch_content_products_raw с курсорной пагинацией
def fetch_content_products_raw() -> list:
    """Выгружает карточки товаров из API Контента с честной курсорной пагинацией через safe_request."""
    if not inspect_wb_token(WB_TOKEN_CONTENT, "WB_TOKEN_CONTENT"):
        raise ValueError("Префлай-валидация токена контента провалена.")
        
    headers = {"Authorization": WB_TOKEN_CONTENT, "Content-Type": "application/json"}
    url = "https://content-api.wildberries.ru/content/v2/get/cards/list"
    
    all_cards = []
    cursor_updated_at = None
    cursor_nm_id = None
    
    while True:
        payload = {
            "settings": {
                "sort": {"ascending": True},
                "cursor": {"limit": 100},
                "filter": {"withPhoto": -1}
            }
        }
        if cursor_updated_at and cursor_nm_id:
            payload["settings"]["cursor"]["updatedAt"] = cursor_updated_at
            payload["settings"]["cursor"]["nmID"] = cursor_nm_id
            
        resp = safe_request("POST", url, headers=headers, json_data=payload, timeout=30)
        data = resp.json()
        
        cards = data.get("cards", [])
        if not cards:
            break
            
        all_cards.extend(cards)
        
        cursor_data = data.get("cursor", {})
        cursor_updated_at = cursor_data.get("updatedAt")
        cursor_nm_id = cursor_data.get("nmID")
        
        if len(cards) < 100 or not cursor_updated_at:
            break
            
        time.sleep(0.6) # Спецификация лимитов API Контента
        
    return all_cards
# --- МОДИФИЦИРОВАННЫЙ БЛОК ЗАПУСКА ТЕСТА ---
try:
    print("=== Старт комплексного теста выгрузки заказов (без буферизации) ===", flush=True)
    raw_orders = fetch_marketplace_orders_raw()
    print(f"=== Тест завершен успешно. Собрано заказов: {len(raw_orders)} ===", flush=True)
    
    # Выведем первые два элемента, если они есть, чтобы убедиться в структуре данных
    if raw_orders:
        print("\n--- Пример структуры полученных данных (первый заказ) ---")
        print(json.dumps(raw_orders[0], indent=2, ensure_ascii=False))
    else:
        print("\n--- Массив заказов пуст, но запросы прошли успешно ---")
except Exception as ex:
    print(f"=== Тест завершен аварийно: {ex} ===", flush=True)

Результат теста: Комплексная выгрузка заказов за скользящее окно выполнена успешно. Собрано 642 заказа, pre-flight валидация и JSON-логирование отработали в штатном режиме.

In [ ]:
import time
import sys
import os
from datetime import date, timedelta

# Имитируем заглушки функций базы данных для Этапа 1
def insert_products_to_staging_mock(data): print(f"[DB MOCK] Отправлено в staging.products: {len(data)} строк")
def insert_orders_to_staging_mock(data): print(f"[DB MOCK] Отправлено в staging.orders: {len(data)} строк")
def insert_sales_to_staging_mock(data): print(f"[DB MOCK] Отправлено в staging.sales: {len(data)} строк")
def insert_stocks_to_staging_mock(data, mode): print(f"[DB MOCK] Отправлено в staging.stocks ({mode}): {len(data)} строк")
def insert_stock_analytics_to_staging_mock(data): print(f"[DB MOCK] Отправлено в staging.stock_analytics: {len(data)} строк")

# Подключаем наш wb_client из папки scripts
ROOT_DIR = r"D:\Documents\Dev\projects\new-bi-project"
if ROOT_DIR not in sys.path:
    sys.path.append(ROOT_DIR)

from scripts.wb_client import (
    write_json_log,
    fetch_content_products_raw,
    fetch_analytics_wb_stocks_raw,
    fetch_analytics_seller_stocks_raw,
    fetch_analytics_products_business_raw,
    safe_request,
    WB_TOKEN_STAT
)

def run_pipeline_optimized_test():
    pipeline_session_id = "pipeline-session-test"
    write_json_log("INFO", pipeline_session_id, "PIPELINE", "START", message="Запуск оптимизированного пайплайна")
    start_time = time.time()
    
    try:
        # 1. Выгрузка каталога товаров (1 запрос)
        write_json_log("INFO", pipeline_session_id, "PIPELINE", "CATALOG_START", message="Выгрузка карточек товаров")
        raw_cards = fetch_content_products_raw()
        insert_products_to_staging_mock(raw_cards)
        
        # Пересчитываем дату для Статистики (один запрос на 5 дней назад)
        start_date_str = f"{(date.today() - timedelta(days=5)).isoformat()}T00:00:00"
        headers_stat = {"Authorization": WB_TOKEN_STAT}
        
        # 2. Выгрузка ЗАКАЗОВ пакетом (1 запрос вместо 6)
        write_json_log("INFO", pipeline_session_id, "PIPELINE", "ORDERS_START", message="Выгрузка заказов единым пакетом")
        url_orders = "https://statistics-api.wildberries.ru/api/v1/supplier/orders"
        resp_orders = safe_request("GET", url_orders, headers=headers_stat, params={"dateFrom": start_date_str, "flag": 1}, timeout=120)
        raw_orders = resp_orders.json() if isinstance(resp_orders.json(), list) else []
        insert_orders_to_staging_mock(raw_orders)
        
        # Ждем 60 секунд ОДИН раз, чтобы гарантированно очистить окно лимита для следующего метода статистики
        write_json_log("INFO", pipeline_session_id, "PIPELINE", "WAIT_LIMIT", message="Пауза 60 сек для очистки лимита статистики перед продажами")
        time.sleep(60.0)
        
        # 3. Выгрузка ПРОДАЖ пакетом (1 запрос вместо 6)
        write_json_log("INFO", pipeline_session_id, "PIPELINE", "SALES_START", message="Выгрузка продаж единым пакетом")
        url_sales = "https://statistics-api.wildberries.ru/api/v1/supplier/sales"
        resp_sales = safe_request("GET", url_sales, headers=headers_stat, params={"dateFrom": start_date_str, "flag": 1}, timeout=120)
        raw_sales = resp_sales.json() if isinstance(resp_sales.json(), list) else []
        insert_sales_to_staging_mock(raw_sales)
        
        # 4. Выгрузка остатков и аналитики (FBO + FBS + Тренды v2)
        nm_ids = list(set([card.get("nmID") for card in raw_cards if card.get("nmID")]))
        chrt_ids = []
        for card in raw_cards:
            for size in card.get("sizes", []):
                c_id = size.get("chrtID")
                if c_id:
                    chrt_ids.append(c_id)
        chrt_ids = list(set(chrt_ids))
        
        if not nm_ids:
            write_json_log("WARNING", pipeline_session_id, "PIPELINE", "CATALOG_EMPTY", message="Каталог пуст")
        else:
            # Выгрузка FBO (Склады WB)
            write_json_log("INFO", pipeline_session_id, "PIPELINE", "STOCKS_FBO_START", message="Выгрузка остатков FBO")
            raw_wb = fetch_analytics_wb_stocks_raw(nm_ids, chrt_ids)
            insert_stocks_to_staging_mock(raw_wb, "wb")
            
            time.sleep(1.0)
            
            # Выгрузка FBS (Склады продавца)
            write_json_log("INFO", pipeline_session_id, "PIPELINE", "STOCKS_FBS_START", message="Выгрузка остатков FBS")
            raw_seller = fetch_analytics_seller_stocks_raw(nm_ids, chrt_ids)
            insert_stocks_to_staging_mock(raw_seller, "seller")
            
            time.sleep(1.0)
            
            # Выгрузка глубокой бизнес-аналитики
            write_json_log("INFO", pipeline_session_id, "PIPELINE", "BUSINESS_ANALYTICS_START", message="Выгрузка бизнес-аналитики остатков")
            raw_business = fetch_analytics_products_business_raw(nm_ids)
            insert_stock_analytics_to_staging_mock(raw_business)
        
        duration = time.time() - start_time
        write_json_log("INFO", pipeline_session_id, "PIPELINE", "SUCCESS", message=f"Оптимизированный пайплайн выполнен за {duration:.2f} сек")
        
    except Exception as e:
        write_json_log("CRITICAL", pipeline_session_id, "PIPELINE", "FAILED", error=str(e), message="Пайплайн упал")

# Запуск
run_pipeline_test = run_pipeline_optimized_test()


Результат теста: Оптимизированный пакетный пайплайн успешно протестирован в памяти за 67.23 сек. Ошибок 429 нет, статус-код 204 по FBS перехвачен корректно.